In [1]:
import pandas as pd
import numpy as np

# # Load the dataset
# data = pd.read_csv('creditcard')
import os;
os.listdir('/kaggle/input/')

['creditcard']

In [2]:
file_path = os.path.join('/kaggle/input/creditcard/', 'creditcard.csv')

# Create the DataFrame
data = pd.read_csv(file_path)

# View the first few rows
data.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
X = data.drop('Class', axis=1)
y = data['Class']

# Perform a stratified train-test split to ensure both sets have the same fraud ratio (0.17%)
# Using 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler,StandardScaler

In [5]:
# Set random seed
np.random.seed(0)
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ("classifier", LogisticRegression(random_state=0))
])

param_grid_lr_cw = param_grid_lr_cw = [
    {
        "classifier": [LogisticRegression(random_state=0)],
        "classifier__penalty": ['l1','l2'],
        "classifier__C": [0.1,0.01,0.001],
        "classifier__solver": ['liblinear'],
        "classifier__class_weight": ['balanced'],
        "classifier__max_iter": [1000,2000]
    }]

# Define Scorer (Good practice for imbalanced data like fraud detection)
# Using AUPRC (Average Precision Score) as suggested in the dataset context. It focuses only on the minority class (fraud).
# AUPRC gives a true picture of performance without being skewed by the large number of easy-to-classify non-fraud cases
scorer = make_scorer(average_precision_score)

grid_lr = GridSearchCV(
    estimator=pipe_lr,
    scoring='average_precision',
    param_grid=param_grid_lr_cw,
    cv=5,
    n_jobs=-1
)

grid_lr.fit(X_train, y_train)
print("Best score: {:.2f}".format(grid_lr.best_score_))
print("Test set score: {:.2f}".format(grid_lr.score(X_test, y_test)))
print("Best parameters: {}".format(grid_lr.best_params_))

Best score: 0.75
Test set score: 0.75
Best parameters: {'classifier': LogisticRegression(random_state=0), 'classifier__C': 0.1, 'classifier__class_weight': 'balanced', 'classifier__max_iter': 1000, 'classifier__penalty': 'l1', 'classifier__solver': 'liblinear'}


In [6]:
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, classification_report

pred = grid_lr.best_estimator_.predict(X_test)
proba = grid_lr.best_estimator_.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC AUC:", roc_auc_score(y_test, proba))
print("AUPRC:", average_precision_score(y_test, proba))
print(classification_report(y_test, pred))

Accuracy: 0.9755802113689829
ROC AUC: 0.9719464917368186
AUPRC: 0.7106100896833404
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962



In [7]:
from sklearn.model_selection import train_test_split

# Assuming 'df' is your DataFrame and 'Class' is your target column
# we take 25% as the subset
sample, _ = train_test_split(
    data, 
    test_size=0.75,       # We toss 95%, keeping 5%
    stratify=data['Class'], # This is the magic line for stratification
    random_state=42       # Ensures you get the same sample every time you run it
)

print(f"Original shape: {data.shape}")
print(f"Sample shape: {sample.shape}")
print(f"Sample Class Distribution:\n{sample['Class'].value_counts(normalize=True)}")

Original shape: (284807, 31)
Sample shape: (71201, 31)
Sample Class Distribution:
Class
0    0.998272
1    0.001728
Name: proportion, dtype: float64


In [8]:
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
X = sample.drop('Class', axis=1)
y = sample['Class']

# Perform a stratified train-test split to ensure both sets have the same fraud ratio (0.17%)
# Using 80% for training and 20% for testing
X_train_sample, X_test_sample, y_train_sample, y_test_sample = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [9]:
pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ("classifier",RandomForestClassifier())
    ])

# Create dictionary with candidate learning algorithms and their hyperparameters
search_space_rf = [{
                 "classifier": [RandomForestClassifier()],
                 "classifier__n_estimators": [100, 300, 500],
                 "classifier__max_features": ['sqrt', 'log2'],
                 "classifier__max_depth": [10,20,30],
                 "classifier__min_samples_split": [2, 5, 10],
                 "classifier__criterion": ['entropy', 'log_loss'],
                 "classifier__n_jobs": [-1],
                 "classifier__class_weight": ['balanced']
                }]
# Create grid search
grid_rf = GridSearchCV(
    estimator=pipe_rf,
    param_grid=search_space_rf,
    scoring='average_precision',
    cv=5,
    n_jobs=-1
)


# Fit grid search
grid_rf.fit(X_train_sample, y_train_sample)

# Return all parameters and components of the pipeline as a dictionary
grid_rf.best_estimator_.get_params()

# View best model
grid_rf.best_estimator_.get_params()["classifier"]

# Predict target vector
grid_rf.predict(X_test_sample)

print("Best accuracy: {:.2f}".format(grid_rf.best_score_))
print("Test set score: {:.2f}".format(grid_rf.score(X_test_sample, y_test_sample)))
print("Best parameters: {}".format(grid_rf.best_params_))

results_rf = grid_rf.cv_results_

Best accuracy: 0.88
Test set score: 0.82
Best parameters: {'classifier': RandomForestClassifier(), 'classifier__class_weight': 'balanced', 'classifier__criterion': 'log_loss', 'classifier__max_depth': 30, 'classifier__max_features': 'sqrt', 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100, 'classifier__n_jobs': -1}


In [10]:
pred = grid_rf.best_estimator_.predict(X_test)
proba = grid_rf.best_estimator_.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC AUC:", roc_auc_score(y_test, proba))
print("AUPRC:", average_precision_score(y_test, proba))
print(classification_report(y_test, pred))

Accuracy: 0.9993153330290369
ROC AUC: 0.9525662913589747
AUPRC: 0.7956814692255846
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.89      0.68      0.77        98

    accuracy                           1.00     56962
   macro avg       0.95      0.84      0.89     56962
weighted avg       1.00      1.00      1.00     56962



In [11]:
!pip install xgboost

In [12]:
from collections import Counter

counter = Counter(y_train_sample)
scale_pos_weight = counter[0] / counter[1]

pipe_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ("classifier",xgb.XGBClassifier())])

# Create dictionary with candidate learning algorithms and their hyperparameters
search_space_xgb = [
               {"classifier": [xgb.XGBClassifier()],
                 "classifier__n_estimators": [10,100,1000],
                 "classifier__max_depth": [3,6,10],
                 "classifier__learning_rate": [0.01, 0.05, 0.1],
                 "classifier__n_jobs": [-1],
                 "classifier__gamma": [0.5,0,1],
                 "classifier__scale_pos_weight": [scale_pos_weight]
                }]

# Create grid search
grid_xgb = GridSearchCV(
    estimator=pipe_xgb,
    param_grid=search_space_xgb,
    scoring='average_precision',
    cv=5,
    n_jobs=-1
)
# Fit grid search
grid_xgb.fit(X_train_sample, y_train_sample)

# Return all parameters and components of the pipeline as a dictionary
grid_xgb.best_estimator_.get_params()

# View best model
grid_xgb.best_estimator_.get_params()["classifier"]

# Predict target vector
grid_xgb.predict(X_test_sample)

print("Best accuracy: {:.2f}".format(grid_xgb.best_score_))
print("Test set score: {:.2f}".format(grid_xgb.score(X_test_sample, y_test_sample)))
print("Best parameters: {}".format(grid_xgb.best_params_))

results_xgb = grid_xgb.cv_results_

Best accuracy: 0.85
Test set score: 0.81
Best parameters: {'classifier': XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...), 'classifier__gamma': 0, 'classifier__learning_rate': 0.1, 'classifier__max_depth': 3, 'classifier__n_estimators': 1000, 'classifier__n_jobs': -1, 'classifier__scale_

In [13]:
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, classification_report
pred = grid_xgb.best_estimator_.predict(X_test)
proba = grid_xgb.best_estimator_.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC AUC:", roc_auc_score(y_test, proba))
print("AUPRC:", average_precision_score(y_test, proba))
print(classification_report(y_test, pred))

Accuracy: 0.9993504441557529
ROC AUC: 0.9760648033833681
AUPRC: 0.8141066534540979
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.84      0.78      0.80        98

    accuracy                           1.00     56962
   macro avg       0.92      0.89      0.90     56962
weighted avg       1.00      1.00      1.00     56962

